In [3]:
# Paramètres [7, 4, 3]
q = 2
F = GF(q)
m = 3
n = 2^m - 1 

# Correction : on force la taille du vecteur à m bits
# 'range(1, n+1)' parcourt les nombres de 1 à 7
# 'Integer(i).bits()' génère la liste des bits, on complète avec des 0 pour atteindre m
H_cols = []
for i in range(1, n + 1):
    b = Integer(i).bits()
    b += [0] * (m - len(b)) # Complétion pour avoir m bits
    H_cols.append(vector(F, b))

H = matrix(F, H_cols).transpose()

print("Matrice de parité H du [7, 4, 3]-code :")
print(H)

# Le reste de l'algorithme (G, codage, décodage) fonctionne à l'identique
G = H.right_kernel().basis_matrix()
print("La matrice generatrice du code est")
print(G)

Matrice de parité H du [7, 4, 3]-code :
[1 0 1 0 1 0 1]
[0 1 1 0 0 1 1]
[0 0 0 1 1 1 1]
La matrice generatrice du code est
[1 0 0 0 0 1 1]
[0 1 0 0 1 0 1]
[0 0 1 0 1 1 0]
[0 0 0 1 1 1 1]


In [4]:
def codage_hamming(message_bits):
    if(len(message_bits)!=4):
        print("Votre ensemble de message n'est pas correcte")
        return
    u = vector(F, message_bits)
    c = u * G
    return c


msg = [1, 0, 1, 1]
mot_code = codage_hamming(msg)
print(f"Message original : {msg}")
print(f"Mot de code envoyé : {mot_code}")

Message original : [1, 0, 1, 1]
Mot de code envoyé : (1, 0, 1, 1, 0, 1, 0)


In [6]:
def decodage_hamming(y_recu):
    if(len(y_recu)!=n):
        print("la longueur du mot recu n'est pas correct")
        return
    y = vector(F, y_recu)
    # Calcul du syndrome S = H * y
    syndrome = H * y
    
    if syndrome == 0:
        print("Aucune erreur détectée.")
        
        # Pour retrouver le message u, on peut utiliser G.solve_left(y) 
        # ou extraire les bits d'information si G est systématique
        message = G.solve_left(y)
        return (y, message)
    else:
        # Conversion du syndrome binaire en indice entier (1-indexed)
        # Comme H est ordonné, l'indice de l'erreur est la valeur du syndrome
        pos_erreur = sum([int(syndrome[i]) * (2^i) for i in range(len(syndrome))])
        print(f"Erreur détectée à la position : {pos_erreur}")
        
        # Correction : on bascule le bit à la position (pos_erreur - 1)
        y_corrige = copy(y)
        y_corrige[pos_erreur - 1] += F(1)
        message = G.solve_left(y_corrige)
        return (y_corrige, message)

# Simulation d'une erreur
y_errone = copy(mot_code)
y_errone[2] += F(1) # Erreur en position 3 (indice 2)

print(f"Mot reçu (avec erreur) : {y_errone}")
mot_retrouve, message = decodage_hamming(y_errone)
print(f"Mot corrigé : {mot_retrouve}")
print(f"Vérification : {mot_retrouve == mot_code}")
print(f"message envoyé est: {message}")

Mot reçu (avec erreur) : (1, 0, 0, 1, 0, 1, 0)
Erreur détectée à la position : 3
Mot corrigé : (1, 0, 1, 1, 0, 1, 0)
Vérification : True
message envoyé est: (1, 0, 1, 1)
